# SCP Data Preparation — Protein-Level for fedRBE

Protein-level extension of the scplainer paper (Vanderaa & Gatto 2024, Figure 6).

- **Datasets:** `leduc2022_plexDIA()`, `derks2022()` (Q-Exactive + timsTOFSCP separately)
- **Level:** protein (aggregated from peptides **within** each lab, then combined)
- **Biology to preserve:** `Celltype`
- **Export:** fedRBE-compatible per-center directories

### Two-stage correction (semi-realistic federated scenario)

**Two benchmark scenarios evaluated:**

| | Local correction (Stage 1) | Federated correction (Stage 2) |
|---|---|---|
| **Scenario 1** (main, harder) | `Label` + `MedianIntensity` | `lab` |
| **Scenario 2** (easier, optimistic) | `Set` + `Label` + `MedianIntensity` | `lab` |

Scenario 1 leaves intra-lab `Set` effects uncorrected, creating a harder cross-site integration problem. Scenario 2 strips most local nuisance first.
A data-driven diagnostic decides per client whether `Set` correction is safe in Scenario 2.

### Key structural difference from the peptide-level notebook

Protein summarization happens **inside each lab object separately**, then labs are combined
via outer-join on canonical protein IDs.

In [212]:
library(scp)
library(scpdata)
library(QFeatures)
library(S4Vectors)
library(SingleCellExperiment)
library(tidyverse)
library(limma)
library(matrixStats)
library(nipals)
library(jsonlite)
library(camprotR)
library(patchwork)

source("../../evaluation_utils/plots_eda.R")
source("../../evaluation_utils/filtering.R")

## Helper functions

In [ ]:
#' Canonicalize protein group IDs so they are stable across labs:
#' split on ";", remove isoform suffixes, sort, rejoin.
canonicalize_protein_ids <- function(x) {
    x <- as.character(x)
    vapply(strsplit(x, ";", fixed = TRUE), function(ids) {
        ids <- trimws(ids)
        ids <- ids[!is.na(ids) & nzchar(ids)]
        ids <- sub("-[0-9]+$", "", ids)
        ids <- sort(unique(ids))
        paste(ids, collapse = ";")
    }, character(1))
}

#' Pad a matrix with NA rows so all matrices have the same row set.
pad_rows <- function(mat, all_rows) {
    out <- matrix(NA_real_, nrow = length(all_rows), ncol = ncol(mat),
                  dimnames = list(all_rows, colnames(mat)))
    out[rownames(mat), ] <- as.matrix(mat)
    as.data.frame(out)
}

#' Generate fedRBE config.yml content
config_template <- function(batch_col_value, covariate_name, is_reference = FALSE) {
    lines <- c(
        "flimmaBatchCorrection:",
        paste0("  batch_col: ", batch_col_value),
        "  covariates:",
        paste0("  - ", covariate_name),
        "  data_filename: intensities_log_UNION.tsv",
        "  design_filename: design.tsv",
        '  design_separator: \"\\t\"',
        "  expression_file_flag: true",
        "  index_col: rowname",
        "  min_samples: 0",
        "  normalizationMethod: null",
        "  position: 0",
        paste0("  reference_batch: ", ifelse(is_reference, "true", "false")),
        '  separator: \"\\t\"',
        "  smpc: true"
    )
    return(lines)
}

#' Export per-center data for fedRBE
export_fedrbe_data <- function(intensities, metadata, intensities_corrected,
                                path_to_data, path_to_after_data,
                                covariate_name = "conditionMonocyte") {
    if (!dir.exists(path_to_data)) dir.create(path_to_data, recursive = TRUE)
    if (!dir.exists(path_to_after_data)) dir.create(path_to_after_data, recursive = TRUE)
    if (!dir.exists(paste0(path_to_data, "00_initial_data/"))) {
        dir.create(paste0(path_to_data, "00_initial_data/"), recursive = TRUE)
    }

    # Save central uncorrected data
    write.table(metadata,
        file = paste0(path_to_data, "00_initial_data/central_batch_info.tsv"),
        sep = "\t", quote = TRUE, row.names = TRUE)
    write.table(intensities %>% as.data.frame() %>% rownames_to_column("rowname"),
        file = paste0(path_to_data, "00_initial_data/central_intensities.tsv"),
        sep = "\t", quote = TRUE, row.names = FALSE)
    write.table(intensities %>% as.data.frame() %>% rownames_to_column("rowname"),
        file = paste0(path_to_data, "00_initial_data/central_intensities_log_UNION.tsv"),
        sep = "\t", quote = TRUE, row.names = FALSE)

    # Design matrix
    design_df <- model.matrix(~ 1 + condition, data = metadata) %>% as.data.frame()

    # Per-center exports
    client_names <- sort(unique(as.character(metadata$lab)))
    reference_client <- client_names[length(client_names)]
    cat("Reference client:", reference_client, "\n")

    for (center in client_names) {
        path_center <- paste0(path_to_data, center, "/")
        if (!dir.exists(path_center)) dir.create(path_center, recursive = TRUE)

        center_metadata <- metadata[metadata$lab == center, ]
        center_intensities <- intensities[, center_metadata$file, drop = FALSE]
        write.table(center_intensities %>% as.data.frame() %>% rownames_to_column('rowname'),
            file = paste0(path_center, "intensities_log_UNION.tsv"),
            sep = "\t", quote = TRUE, row.names = FALSE, col.names = TRUE)

        center_rows <- rownames(center_metadata)
        design_center <- design_df[center_rows, , drop = FALSE]
        design_center$batch <- as.character(metadata[center_rows, "batch"])
        design_center$condition <- as.character(metadata[center_rows, "condition"])

        write.table(design_center %>% rownames_to_column('file'),
            file = paste0(path_center, "design.tsv"),
            sep = "\t", quote = TRUE, row.names = FALSE, col.names = TRUE)

        is_ref <- (center == reference_client)
        writeLines(config_template("batch", covariate_name, is_ref),
                   paste0(path_center, "config.yml"))
        cat("  Center:", center, "- batches:",
            length(unique(as.character(center_metadata$batch))),
            "- reference:", is_ref, "\n")
    }

    # Save R-corrected data
    write.table(intensities_corrected %>% as.data.frame() %>% rownames_to_column('rowname'),
        file = paste0(path_to_after_data, "intensities_log_Rcorrected_UNION.tsv"),
        sep = "\t", quote = TRUE, row.names = FALSE, col.names = TRUE)

    # Write datainfo.json
    cohorts_list <- lapply(client_names, function(center) {
        list(name = center, folder = center)
    })
    datainfo <- list(
        covariates = c(covariate_name),
        datafile = list(
            filename = "intensities_log_UNION.tsv",
            separator = "\t",
            rotation = "features x samples",
            samplename_column = NULL,
            featurename_column = "rowname"
        ),
        designfile = list(
            filename = "design.tsv",
            separator = "\t",
            rotation = "samples x features",
            samplename_column = "file",
            featurename_column = NULL
        ),
        cohorts = cohorts_list
    )
    writeLines(toJSON(datainfo, pretty = TRUE, auto_unbox = TRUE),
        paste0(path_to_data, "datainfo.json"))
    cat("  datainfo.json written to", paste0(path_to_data, "datainfo.json"), "\n")
}

---
# Part 1: Minimal Processing Per Dataset

Same minimal processing as the scplainer repo — no extra peptide filters.

## 1.1) Leduc 2022 plexDIA — precursors → peptides → proteins

Following the scplainer paper/repo logic:
- Remove pre-aggregated assays, keep only `Ms1Extracted`
- Zero → NA, remove contaminants
- QC filtering (MedianIntensity, passQC)
- Aggregate precursors → peptides by `Stripped.Sequence`
- Log-transform
- **Aggregate peptides → proteins by canonical `Protein.Ids` within this lab**
- Keep only Melanoma samples (as in the original paper)

In [214]:
leduc <- leduc2022_plexDIA()

## Keep only required data
leduc <- removeAssay(leduc, c("peptides", "proteins"))

requiredRowData <- c(
    "Protein.Group",
    "Protein.Ids",
    "Protein.Names",
    "Genes",
    "First.Protein.Description",
    "Proteotypic",
    "Stripped.Sequence",
    "Modified.Sequence",
    "Precursor.Charge",
    "Precursor.Id"
)
leduc <- selectRowData(leduc, requiredRowData)

## Zeros -> NA
leduc <- zeroIsNA(leduc, i = names(leduc))

## Remove contaminants
contaminants <- paste(get_ccp_crap(), collapse = "|")
for (i in names(leduc)) {
    rowData(leduc[[i]])$is_contaminant <- grepl(contaminants, rowData(leduc[[i]])$Protein.Ids)
}
leduc <- filterFeatures(leduc, ~ !is_contaminant)

## Sample QC metrics
leduc <- countUniqueFeatures(
    leduc,
    i = "Ms1Extracted",
    groupBy = "Stripped.Sequence",
    colDataName = "NumberPeptides"
)

logInt <- log(assay(leduc[["Ms1Extracted"]]))
MedianIntensity <- colMedians(logInt, na.rm = TRUE)
names(MedianIntensity) <- colnames(leduc[["Ms1Extracted"]])
colData(leduc)[names(MedianIntensity), "MedianIntensity"] <- MedianIntensity

leduc <- medianCVperCell(
    leduc,
    i = "Ms1Extracted",
    groupBy = "Protein.Group",
    nobs = 3,
    na.rm = TRUE,
    colDataName = "MedianCV",
    norm = "SCoPE2"
)

## Same sample filter as in the repo — keep only Melanoma cells passing QC
leduc$passQC <- leduc$MedianIntensity > 9.5 & leduc$SampleType == "Melanoma"
leduc <- subsetByColData(leduc, leduc$passQC)

## Aggregate precursors -> peptides
leduc <- aggregateFeatures(
    leduc,
    i = "Ms1Extracted",
    fcol = "Stripped.Sequence",
    name = "peptides",
    fun = colMedians,
    na.rm = TRUE
)

## Log-transform
leduc <- logTransform(leduc, i = "peptides", name = "peptides_log")

cat("Leduc plexDIA peptides_log:", dim(getWithColData(leduc, "peptides_log")), "\n")

see ?scpdata and browseVignettes('scpdata') for documentation

loading from cache

Warning message:
“'experiments' dropped; see 'drops()'”
harmonizing input:
  removing 209 sampleMap rows not in names(experiments)
  removing 1 colData rownames not in sampleMap 'primary'

'is_contaminant' found in 46 out of 46 assay(s).

Your quantitative data contain missing values. Please read the relevant
section(s) in the aggregateFeatures manual page regarding the effects
of missing values on data aggregation.

Aggregated: 1/1



Leduc plexDIA peptides_log: 2515 106 


### Aggregate peptides → proteins (within Leduc)

In [215]:
## Add canonical protein key
rowData(leduc[["peptides_log"]])$ProteinKey <- canonicalize_protein_ids(
    rowData(leduc[["peptides_log"]])$Protein.Ids
)

## Aggregate peptides -> proteins (default: robustSummary)
leduc <- aggregateFeatures(
    leduc,
    i = "peptides_log",
    fcol = "ProteinKey",
    name = "proteins_log",
    fun = colMedians,
    na.rm = TRUE
)

cat("Leduc plexDIA proteins_log:", dim(getWithColData(leduc, "proteins_log")), "\n")

Your quantitative data contain missing values. Please read the relevant
section(s) in the aggregateFeatures manual page regarding the effects
of missing values on data aggregation.

Aggregated: 1/1



Leduc plexDIA proteins_log: 912 106 


In [216]:
## Extract protein matrix and metadata for Leduc
leduc_prot_sce <- getWithColData(leduc, "proteins_log")
leduc_prot_mat <- as.matrix(assay(leduc_prot_sce))

leduc_prot_meta <- data.frame(
    Celltype = leduc_prot_sce$SampleType,
    Set = as.character(colData(leduc_prot_sce)[[1]]),
    Label = as.character(leduc_prot_sce$Label),
    MedianIntensity = leduc_prot_sce$MedianIntensity,
    dataset = "leduc_plexDIA",
    row.names = colnames(leduc_prot_sce),
    stringsAsFactors = FALSE
)

# Replace missing/empty labels with "none"
leduc_prot_meta$Label[is.na(leduc_prot_meta$Label) | leduc_prot_meta$Label == ""] <- "none"

## Prefix sample names to avoid collisions
colnames(leduc_prot_mat) <- paste0("leduc__", colnames(leduc_prot_mat))
rownames(leduc_prot_meta) <- paste0("leduc__", rownames(leduc_prot_meta))

cat("Leduc protein matrix:", dim(leduc_prot_mat), "\n")
cat("Leduc metadata:", dim(leduc_prot_meta), "\n")
head(leduc_prot_meta)

Leduc protein matrix: 912 106 
Leduc metadata: 106 5 


,Celltype,Set,Label,MedianIntensity,dataset
,<chr>,<chr>,<chr>,<dbl>,<chr>
leduc__D..AL.AL.wAL_plexMel01.raw.0,Melanoma,wAL_plexMel01,0,10.42631,leduc_plexDIA
leduc__D..AL.AL.wAL_plexMel01.raw.4,Melanoma,wAL_plexMel01,4,10.08633,leduc_plexDIA
leduc__D..AL.AL.wAL_plexMel01.raw.8,Melanoma,wAL_plexMel01,8,10.29381,leduc_plexDIA
leduc__D..AL.AL.wAL_plexMel03.raw.0,Melanoma,wAL_plexMel03,0,10.14752,leduc_plexDIA
leduc__D..AL.AL.wAL_plexMel03.raw.4,Melanoma,wAL_plexMel03,4,10.05808,leduc_plexDIA
leduc__D..AL.AL.wAL_plexMel05.raw.0,Melanoma,wAL_plexMel05,0,10.08856,leduc_plexDIA


In [217]:
# Save intermediate data
if (!dir.exists("before/00_initial_data")) dir.create("before/00_initial_data", recursive = TRUE)
saveRDS(leduc_prot_mat, "./before/00_initial_data/leduc_proteins_log_matrix.rds")
saveRDS(leduc_prot_meta, "./before/00_initial_data/leduc_proteins_log_metadata.rds")

## 1.2) Derks 2022 — precursors → peptides → proteins (Q-Exactive and timsTOFSCP separately)

Following the scplainer paper/repo logic:
- Remove bulk assays and pre-aggregated proteins
- Zero → NA, remove contaminants + keratins
- QC filtering per instrument (separate thresholds)
- Aggregate precursors → peptides by `Stripped.Sequence`
- Log-transform
- **Split by instrument, then aggregate peptides → proteins within each instrument/lab**

In [218]:
derks <- derks2022()

## Keep only required data
assaysToRemove <- c(
    grep("119[456]", names(derks), value = TRUE),
    "bulk_prec_extracted",
    "proteins"
)
derks <- removeAssay(derks, assaysToRemove)

requiredRowData <- c(
    "Protein.Group",
    "Protein.Ids",
    "Protein.Names",
    "Genes",
    "First.Protein.Description",
    "Proteotypic",
    "Stripped.Sequence",
    "Modified.Sequence",
    "Precursor.Charge",
    "Precursor.Id"
)
derks <- selectRowData(derks, requiredRowData)

## Reformat annotations + zeros -> NA
derks$Celltype <- sub("_t", "", derks$Celltype)
derks <- zeroIsNA(derks, i = names(derks))

## Remove contaminants and keratins
contaminants <- paste(get_ccp_crap(), collapse = "|")
for (i in names(derks)) {
    rowData(derks[[i]])$is_contaminant <- grepl(contaminants, rowData(derks[[i]])$Protein.Ids)
    rowData(derks[[i]])$is_keratin <- grepl("KRT|^K\\d.\\d", rowData(derks[[i]])$Protein.Names)
}
derks <- filterFeatures(derks, ~ !is_contaminant & !is_keratin)

## Sample QC metrics
sel <- grep("extracted", names(derks), value = TRUE)

derks <- countUniqueFeatures(
    derks,
    i = sel,
    groupBy = "Stripped.Sequence",
    colDataName = "NumberPeptides"
)

MedianIntensity <- lapply(experiments(derks)[sel], function(x) {
    out <- colMedians(log(assay(x)), na.rm = TRUE)
    names(out) <- colnames(x)
    out
})
names(MedianIntensity) <- NULL
MedianIntensity <- unlist(MedianIntensity)
colData(derks)[names(MedianIntensity), "MedianIntensity"] <- MedianIntensity

derks <- medianCVperCell(
    derks,
    i = sel,
    groupBy = "Protein.Group",
    nobs = 3,
    na.rm = TRUE,
    colDataName = "MedianCV",
    norm = "SCoPE2"
)

## Instrument-specific sample filters (same as in the repo)
derks$passQC <- derks$Instrument != "Q-Exactive" |
    (derks$MedianIntensity > 9.4 &
     derks$NumberPeptides > 750 &
     derks$Celltype != "Neg")
derks$passQC[is.na(derks$passQC)] <- FALSE
derks <- subsetByColData(derks, derks$passQC)

derks$passQC <- derks$Instrument != "timsTOFSCP" |
    (derks$MedianIntensity > 6.5 &
     !grepl("DB$", derks$Celltype))
derks$passQC[is.na(derks$passQC)] <- FALSE
derks <- subsetByColData(derks, derks$passQC)

## Aggregate precursor -> peptide within each extracted assay
derks <- aggregateFeatures(
    derks,
    i = sel,
    fcol = "Stripped.Sequence",
    name = paste0("peptides_", sel),
    fun = colMedians,
    na.rm = TRUE
)

## Join assays and log-transform
derks <- joinAssays(derks, i = paste0("peptides_", sel), name = "peptides")
derks <- logTransform(derks, i = "peptides", name = "peptides_log")

cat("Derks peptides_log:", dim(getWithColData(derks, "peptides_log")), "\n")

see ?scpdata and browseVignettes('scpdata') for documentation

loading from cache

Warning message:
“'experiments' dropped; see 'drops()'”
harmonizing input:
  removing 182 sampleMap rows not in names(experiments)
  removing 9 colData rownames not in sampleMap 'primary'

'is_contaminant' found in 61 out of 61 assay(s).'is_keratin' found in 61 out of 61 assay(s).

Your quantitative data contain missing values. Please read the relevant
section(s) in the aggregateFeatures manual page regarding the effects
of missing values on data aggregation.

Aggregated: 1/2

Your quantitative data contain missing values. Please read the relevant
section(s) in the aggregateFeatures manual page regarding the effects
of missing values on data aggregation.

Aggregated: 2/2



Derks peptides_log: 6679 122 


### 1.2.1 Split Derks by instrument, aggregate to proteins separately

In [ ]:
## Split by instrument BEFORE protein aggregation
derks_qe <- subsetByColData(derks, derks$Instrument == "Q-Exactive")
derks_tims <- subsetByColData(derks, derks$Instrument == "timsTOFSCP")

## --- Q-Exactive: aggregate peptides -> proteins (default: robustSummary) ---
rowData(derks_qe[["peptides_log"]])$ProteinKey <- canonicalize_protein_ids(
    rowData(derks_qe[["peptides_log"]])$Protein.Ids
)
derks_qe <- aggregateFeatures(
    derks_qe,
    i = "peptides_log",
    fcol = "ProteinKey",
    name = "proteins_log",
    fun = colMedians,
    na.rm = TRUE

)
cat("Derks Q-Exactive proteins_log:", dim(getWithColData(derks_qe, "proteins_log")), "\n")

## --- timsTOFSCP: aggregate peptides -> proteins (default: robustSummary) ---
rowData(derks_tims[["peptides_log"]])$ProteinKey <- canonicalize_protein_ids(
    rowData(derks_tims[["peptides_log"]])$Protein.Ids
)
derks_tims <- aggregateFeatures(
    derks_tims,
    i = "peptides_log",
    fcol = "ProteinKey",
    name = "proteins_log",
    fun = colMedians,
    na.rm = TRUE
)
cat("Derks timsTOFSCP proteins_log:", dim(getWithColData(derks_tims, "proteins_log")), "\n")

Your quantitative data contain missing values. Please read the relevant
section(s) in the aggregateFeatures manual page regarding the effects
of missing values on data aggregation.

Aggregated: 1/1



Derks Q-Exactive proteins_log: 1841 97 


Your quantitative data contain missing values. Please read the relevant
section(s) in the aggregateFeatures manual page regarding the effects
of missing values on data aggregation.

Aggregated: 1/1



Derks timsTOFSCP proteins_log: 1841 25 


In [220]:
## Extract protein matrix and metadata for Derks Q-Exactive
derks_qe_prot_sce <- getWithColData(derks_qe, "proteins_log")
derks_qe_prot_mat <- as.matrix(assay(derks_qe_prot_sce))

## Harmonise cell type labels
derks_qe_prot_sce$Celltype <- ifelse(derks_qe_prot_sce$Celltype == "U-937", "Monocyte",
                                      derks_qe_prot_sce$Celltype)

derks_qe_prot_meta <- data.frame(
    Celltype = derks_qe_prot_sce$Celltype,
    Set = as.character(derks_qe_prot_sce$Set),
    Label = as.character(derks_qe_prot_sce$Label),
    MedianIntensity = derks_qe_prot_sce$MedianIntensity,
    dataset = "derks_QExactive",
    row.names = colnames(derks_qe_prot_sce),
    stringsAsFactors = FALSE
)

colnames(derks_qe_prot_mat) <- paste0("derks_qe__", colnames(derks_qe_prot_mat))
rownames(derks_qe_prot_meta) <- paste0("derks_qe__", rownames(derks_qe_prot_meta))

cat("Derks Q-Exactive protein matrix:", dim(derks_qe_prot_mat), "\n")

## Extract protein matrix and metadata for Derks timsTOFSCP
derks_tims_prot_sce <- getWithColData(derks_tims, "proteins_log")
derks_tims_prot_mat <- as.matrix(assay(derks_tims_prot_sce))

## Harmonise cell type labels
derks_tims_prot_sce$Celltype <- ifelse(derks_tims_prot_sce$Celltype == "U-937", "Monocyte",
                                        derks_tims_prot_sce$Celltype)

derks_tims_prot_meta <- data.frame(
    Celltype = derks_tims_prot_sce$Celltype,
    Set = as.character(derks_tims_prot_sce$Set),
    Label = as.character(derks_tims_prot_sce$Label),
    MedianIntensity = derks_tims_prot_sce$MedianIntensity,
    dataset = "derks_timsTOFSCP",
    row.names = colnames(derks_tims_prot_sce),
    stringsAsFactors = FALSE
)

colnames(derks_tims_prot_mat) <- paste0("derks_tims__", colnames(derks_tims_prot_mat))
rownames(derks_tims_prot_meta) <- paste0("derks_tims__", rownames(derks_tims_prot_meta))

cat("Derks timsTOFSCP protein matrix:", dim(derks_tims_prot_mat), "\n")

Derks Q-Exactive protein matrix: 1841 97 
Derks timsTOFSCP protein matrix: 1841 25 


In [221]:
# Save intermediate data
saveRDS(derks_qe_prot_mat, "./before/00_initial_data/derks_qe_proteins_log_matrix.rds")
saveRDS(derks_qe_prot_meta, "./before/00_initial_data/derks_qe_proteins_log_metadata.rds")
saveRDS(derks_tims_prot_mat, "./before/00_initial_data/derks_tims_proteins_log_matrix.rds")
saveRDS(derks_tims_prot_meta, "./before/00_initial_data/derks_tims_proteins_log_metadata.rds")

In [222]:
# Free memory from Part 1 processing objects
rm(leduc, leduc_prot_sce, leduc_prot_mat, leduc_prot_meta,
   derks, derks_qe, derks_tims,
   derks_qe_prot_sce, derks_qe_prot_mat, derks_qe_prot_meta,
   derks_tims_prot_sce, derks_tims_prot_mat, derks_tims_prot_meta)
gc()

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,12187504,650.9,21211491,1132.9,21211491,1132.9
Vcells,44700114,341.1,84241109,642.8,84241107,642.8


---
# Part 2: Combine protein matrices across labs

Union of canonical protein IDs across the 3 labs, padding missing proteins with NA.

In [223]:
# Read all matrices and metadata back in (to simulate re-import)
leduc_prot_mat <- readRDS("./before/00_initial_data/leduc_proteins_log_matrix.rds")
leduc_prot_meta <- readRDS("./before/00_initial_data/leduc_proteins_log_metadata.rds")

derks_qe_prot_mat <- readRDS("./before/00_initial_data/derks_qe_proteins_log_matrix.rds")
derks_qe_prot_meta <- readRDS("./before/00_initial_data/derks_qe_proteins_log_metadata.rds")

derks_tims_prot_mat <- readRDS("./before/00_initial_data/derks_tims_proteins_log_matrix.rds")
derks_tims_prot_meta <- readRDS("./before/00_initial_data/derks_tims_proteins_log_metadata.rds")

In [224]:
all_proteins <- Reduce(union, list(
    rownames(leduc_prot_mat),
    rownames(derks_qe_prot_mat),
    rownames(derks_tims_prot_mat)
))

leduc_prot_mat_u     <- pad_rows(leduc_prot_mat, all_proteins)
derks_qe_prot_mat_u  <- pad_rows(derks_qe_prot_mat, all_proteins)
derks_tims_prot_mat_u <- pad_rows(derks_tims_prot_mat, all_proteins)

intensities <- cbind(leduc_prot_mat_u, derks_qe_prot_mat_u, derks_tims_prot_mat_u)
int_meta <- dplyr::bind_rows(
    leduc_prot_meta,
    derks_qe_prot_meta,
    derks_tims_prot_meta
)
stopifnot(identical(colnames(intensities), rownames(int_meta)))

# Remove all-NA rows and columns
intensities <- intensities[rowSums(is.na(intensities)) < ncol(intensities), ]
intensities <- intensities[, colSums(is.na(intensities)) < nrow(intensities), drop = FALSE]
int_meta <- int_meta[colnames(intensities), , drop = FALSE]

# Map to fedRBE-compatible metadata columns
metadata <- data.frame(
    file = rownames(int_meta),
    batch = paste0(int_meta$dataset, "__", int_meta$Set),
    condition = int_meta$Celltype,
    lab = int_meta$dataset,
    label = int_meta$Label,
    MedianIntensity = int_meta$MedianIntensity,
    Set = int_meta$Set,
    stringsAsFactors = FALSE
)
rownames(metadata) <- metadata$file

cat("Combined protein intensities:", dim(intensities), "\n")
cat("Labs:", unique(metadata$lab), "\n")
cat("Conditions:", unique(metadata$condition), "\n")

Combined protein intensities: 1708 228 
Labs: leduc_plexDIA derks_QExactive derks_timsTOFSCP 
Conditions: Melanoma PDAC Monocyte 


### Dataset summary

In [225]:
# Summary by Center and Batch
lab_summary <- metadata %>%
    dplyr::group_by(lab, batch) %>%
    dplyr::summarise(
        `Total samples` = dplyr::n(),
        `Total features` = sum(rowSums(!is.na(intensities[, file, drop = FALSE])) > 0),
        .groups = "drop"
    )

sample_counts <- metadata %>%
    dplyr::group_by(lab, batch, condition) %>%
    dplyr::summarise(count = n(), .groups = "drop") %>%
    tidyr::pivot_wider(names_from = condition, values_from = count, values_fill = 0)

final_summary <- lab_summary %>%
    dplyr::left_join(sample_counts, by = c("lab", "batch")) %>%
    dplyr::rename(Center = lab, Batch = batch) %>%
    dplyr::select(Center, Batch, `Total features`, `Total samples`, everything())

print("Dataset Summary by Center and Batch:")
knitr::kable(final_summary, format = "markdown")

[1] "Dataset Summary by Center and Batch:"




|Center           |Batch                        | Total features| Total samples| Melanoma| PDAC| Monocyte|
|:----------------|:----------------------------|--------------:|-------------:|--------:|----:|--------:|
|derks_QExactive  |derks_QExactive__1           |            649|             2|        1|    1|        0|
|derks_QExactive  |derks_QExactive__10          |            661|             2|        1|    1|        0|
|derks_QExactive  |derks_QExactive__11          |            581|             3|        1|    1|        1|
|derks_QExactive  |derks_QExactive__12          |            782|             2|        1|    1|        0|
|derks_QExactive  |derks_QExactive__13          |            634|             2|        0|    1|        1|
|derks_QExactive  |derks_QExactive__14          |            710|             2|        1|    1|        0|
|derks_QExactive  |derks_QExactive__15          |            816|             2|        1|    1|        0|
|derks_QExactive  |derks_QExactive_

In [226]:
# Summary by Center
lab_summary2 <- metadata %>%
    dplyr::group_by(lab) %>%
    dplyr::summarise(
        `Total samples` = dplyr::n(),
        `Total features` = sum(rowSums(!is.na(intensities[, file, drop = FALSE])) > 0),
        .groups = "drop"
    )

sample_counts2 <- metadata %>%
    dplyr::group_by(lab, condition) %>%
    dplyr::summarise(count = dplyr::n(), .groups = "drop") %>%
    tidyr::pivot_wider(names_from = condition, values_from = count, values_fill = 0)

final_summary2 <- lab_summary2 %>%
    dplyr::left_join(sample_counts2, by = "lab") %>%
    dplyr::rename(Center = lab) %>%
    dplyr::select(Center, `Total features`, `Total samples`, everything())

print("Dataset Summary by Center:")
knitr::kable(final_summary2, format = "markdown")

[1] "Dataset Summary by Center:"




|Center           | Total features| Total samples| Melanoma| Monocyte| PDAC|
|:----------------|--------------:|-------------:|--------:|--------:|----:|
|derks_QExactive  |           1220|            97|       38|       26|   33|
|derks_timsTOFSCP |           1485|            25|        9|       10|    6|
|leduc_plexDIA    |            899|           106|      106|        0|    0|

---
# Part 3: Filtering

### Sample-level outlier filtering within each lab

Combine robust QC metrics, correlation, and PCA distance inside each lab.
Because `condition` is the biology to preserve, automatic removal requires both
technical QC support and condition-aware structural support.


In [227]:
flag_by_mad <- function(x, nmads = 3, direction = c("both", "lower", "higher")) {
  direction <- match.arg(direction)
  med <- median(x, na.rm = TRUE)
  s <- mad(x, na.rm = TRUE)

  if (is.na(s) || s == 0) {
    return(rep(FALSE, length(x)))
  }

  lower <- med - nmads * s
  upper <- med + nmads * s

  if (direction == "both") {
    x < lower | x > upper
  } else if (direction == "lower") {
    x < lower
  } else {
    x > upper
  }
}

median_cor_subset <- function(expr_mat, groups) {
  out <- rep(NA_real_, ncol(expr_mat))
  names(out) <- colnames(expr_mat)

  for (g in unique(groups)) {
    idx <- which(groups == g)
    if (length(idx) < 3) {
      next
    }

    cor_mat <- suppressWarnings(cor(
      expr_mat[, idx, drop = FALSE],
      use = "pairwise.complete.obs",
      method = "pearson"
    ))
    diag(cor_mat) <- NA_real_
    out[idx] <- apply(cor_mat, 2, median, na.rm = TRUE)
  }

  out
}

robust_covariance <- function(x) {
  if (requireNamespace("robustbase", quietly = TRUE)) {
    fit <- robustbase::covMcd(x)
    return(list(center = fit$center, cov = fit$cov, method = "robustbase::covMcd"))
  }

  if (requireNamespace("MASS", quietly = TRUE)) {
    fit <- MASS::cov.rob(x, method = "mcd")
    return(list(center = fit$center, cov = fit$cov, method = "MASS::cov.rob(method='mcd')"))
  }

  stop("Need either the robustbase or MASS package for PCA-distance outlier detection.")
}

make_complete_for_pca <- function(mat, min_detect_frac = 0.20) {
  keep <- rowMeans(!is.na(mat)) >= min_detect_frac
  mat_pca <- as.matrix(mat[keep, , drop = FALSE])

  if (!nrow(mat_pca)) {
    return(mat_pca)
  }

  row_meds <- apply(mat_pca, 1, median, na.rm = TRUE)
  for (i in seq_len(nrow(mat_pca))) {
    nas <- is.na(mat_pca[i, ])
    if (any(nas)) {
      mat_pca[i, nas] <- row_meds[i]
    }
  }

  mat_pca
}

detect_lab_outliers <- function(
  expr_mat,
  meta_lab,
  nmads = 3,
  min_detect_frac = 0.20,
  p_cut = 0.001,
  max_pcs = 5L,
  inspect_n_flags = 2L
) {
  stopifnot(all(colnames(expr_mat) %in% meta_lab$file))

  sample_qc <- tibble(
    file = colnames(expr_mat),
    n_detected = colSums(!is.na(expr_mat)),
    pct_missing = colMeans(is.na(expr_mat)),
    median_intensity = apply(expr_mat, 2, median, na.rm = TRUE),
    mad_intensity = apply(expr_mat, 2, mad, na.rm = TRUE)
  ) %>%
    left_join(meta_lab, by = "file")

  sample_cor <- suppressWarnings(cor(
    expr_mat,
    use = "pairwise.complete.obs",
    method = "pearson"
  ))
  diag(sample_cor) <- NA_real_

  median_cor_condition <- median_cor_subset(expr_mat, sample_qc$condition)
  out_median_cor_condition <- flag_by_mad(
    median_cor_condition,
    nmads = nmads,
    direction = "lower"
  )
  out_median_cor_condition[is.na(out_median_cor_condition)] <- FALSE

  sample_qc <- sample_qc %>% mutate(
    median_cor = apply(sample_cor, 2, median, na.rm = TRUE),
    median_cor_condition = median_cor_condition,
    out_n_detected = flag_by_mad(n_detected, nmads = nmads, direction = "lower"),
    out_pct_missing = flag_by_mad(pct_missing, nmads = nmads, direction = "higher"),
    out_median_intensity = flag_by_mad(median_intensity, nmads = nmads, direction = "both"),
    out_mad_intensity = flag_by_mad(mad_intensity, nmads = nmads, direction = "both"),
    out_median_cor = flag_by_mad(median_cor, nmads = nmads, direction = "lower"),
    out_median_cor_condition = out_median_cor_condition,
    robust_md2 = NA_real_,
    pca_cutoff = NA_real_,
    pca_method = NA_character_,
    out_pca_md = FALSE
  )

  n_samples <- ncol(expr_mat)
  mat_pca <- make_complete_for_pca(expr_mat, min_detect_frac = min_detect_frac)

  if (nrow(mat_pca) >= 2 && n_samples >= 6) {
    pca <- prcomp(t(mat_pca), center = TRUE, scale. = TRUE)
    non_zero_var <- which(apply(pca$x, 2, var, na.rm = TRUE) > 0)
    npc <- min(max_pcs, length(non_zero_var), floor((n_samples - 1) / 2))

    if (npc >= 2) {
      pc_scores <- pca$x[, non_zero_var[seq_len(npc)], drop = FALSE]
      fit <- tryCatch(robust_covariance(pc_scores), error = function(e) NULL)
      cov_det <- if (is.null(fit)) {
        NA_real_
      } else {
        tryCatch(det(fit$cov), error = function(e) NA_real_)
      }

      if (!is.null(fit) && is.finite(cov_det) && cov_det > 0) {
        md2 <- mahalanobis(pc_scores, center = fit$center, cov = fit$cov)
        cutoff <- qchisq(1 - p_cut, df = npc)
        sample_qc$robust_md2 <- md2[match(sample_qc$file, rownames(pc_scores))]
        sample_qc$pca_cutoff <- cutoff
        sample_qc$pca_method <- fit$method
        sample_qc$out_pca_md <- sample_qc$robust_md2 > cutoff
      }
    }
  }

  sample_qc %>% mutate(
    n_outlier_flags = out_n_detected +
      out_pct_missing +
      out_median_intensity +
      out_mad_intensity +
      out_median_cor +
      out_pca_md,
    qc_support = out_n_detected | out_pct_missing | out_median_intensity | out_mad_intensity,
    structure_support = out_pca_md | out_median_cor_condition,
    inspect_outlier = n_outlier_flags >= inspect_n_flags | out_median_cor_condition,
    remove_outlier = qc_support & structure_support
  )
}

expr_mat <- intensities
meta <- metadata %>%
  mutate(
    file = as.character(file),
    lab = as.character(lab),
    condition = as.character(condition)
  )

sample_qc <- bind_rows(lapply(split(meta, meta$lab), function(meta_lab) {
  expr_lab <- as.matrix(expr_mat[, meta_lab$file, drop = FALSE])
  detect_lab_outliers(expr_lab, meta_lab)
}))

sample_qc %>%
  count(lab, inspect_outlier, remove_outlier, name = "n_samples")

sample_qc %>%
  filter(inspect_outlier) %>%
  arrange(lab, desc(remove_outlier), desc(n_outlier_flags), median_cor_condition, median_cor, desc(pct_missing)) %>%
  select(
    file,
    lab,
    condition,
    batch,
    Set,
    n_detected,
    pct_missing,
    median_intensity,
    mad_intensity,
    median_cor,
    median_cor_condition,
    robust_md2,
    n_outlier_flags,
    out_n_detected,
    out_pct_missing,
    out_median_intensity,
    out_mad_intensity,
    out_median_cor,
    out_median_cor_condition,
    out_pca_md,
    remove_outlier
  )


lab,inspect_outlier,remove_outlier,n_samples
<chr>,<lgl>,<lgl>,<int>
derks_QExactive,FALSE,FALSE,83
derks_QExactive,TRUE,FALSE,14
derks_timsTOFSCP,FALSE,FALSE,23
derks_timsTOFSCP,TRUE,FALSE,2
leduc_plexDIA,FALSE,FALSE,90
leduc_plexDIA,TRUE,FALSE,9
leduc_plexDIA,TRUE,TRUE,7


file,lab,condition,batch,Set,n_detected,pct_missing,median_intensity,mad_intensity,median_cor,⋯,robust_md2,n_outlier_flags,out_n_detected,out_pct_missing,out_median_intensity,out_mad_intensity,out_median_cor,out_median_cor_condition,out_pca_md,remove_outlier
<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<int>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>
derks_qe__F..JD.plexDIA.nPOP.wJD1150.raw.0,derks_QExactive,Melanoma,derks_QExactive__5,5,483,0.7172131,13.73667,1.604417,0.5303212,⋯,218.133720,2,FALSE,FALSE,FALSE,FALSE,TRUE,TRUE,TRUE,FALSE
derks_qe__F..JD.plexDIA.nPOP.wJD1182.raw.4,derks_QExactive,Melanoma,derks_QExactive__37,37,470,0.7248244,13.86765,1.439784,0.5472010,⋯,178.397685,2,FALSE,FALSE,FALSE,FALSE,TRUE,TRUE,TRUE,FALSE
derks_qe__F..JD.plexDIA.nPOP.wJD1177.raw.0,derks_QExactive,Melanoma,derks_QExactive__32,32,472,0.7236534,13.81591,1.609115,0.5441505,⋯,223.983257,2,FALSE,FALSE,FALSE,FALSE,TRUE,TRUE,TRUE,FALSE
derks_qe__F..JD.plexDIA.nPOP.wJD1192.raw.4,derks_QExactive,Melanoma,derks_QExactive__47,47,519,0.6961358,13.74246,1.443765,0.5620515,⋯,218.789479,2,FALSE,FALSE,FALSE,FALSE,TRUE,TRUE,TRUE,FALSE
derks_qe__F..JD.plexDIA.nPOP.wJD1151.raw.8,derks_QExactive,Melanoma,derks_QExactive__6,6,466,0.7271663,14.09235,1.419214,0.5676678,⋯,203.011588,2,FALSE,FALSE,FALSE,FALSE,TRUE,TRUE,TRUE,FALSE
derks_qe__F..JD.plexDIA.nPOP.wJD1185.raw.0,derks_QExactive,Melanoma,derks_QExactive__40,40,410,0.7599532,13.58691,1.553197,0.5683952,⋯,134.559210,2,FALSE,FALSE,FALSE,FALSE,TRUE,TRUE,TRUE,FALSE
derks_qe__F..JD.plexDIA.nPOP.wJD1153.raw.0,derks_QExactive,Melanoma,derks_QExactive__8,8,466,0.7271663,13.28045,1.517141,0.5805358,⋯,175.600007,2,FALSE,FALSE,FALSE,FALSE,TRUE,TRUE,TRUE,FALSE
derks_qe__F..JD.plexDIA.nPOP.wJD1147.raw.0,derks_QExactive,Melanoma,derks_QExactive__2,2,529,0.6902810,13.76109,1.557352,0.5853684,⋯,203.901951,2,FALSE,FALSE,FALSE,FALSE,TRUE,TRUE,TRUE,FALSE
derks_qe__F..JD.plexDIA.nPOP.wJD1169.raw.4,derks_QExactive,Melanoma,derks_QExactive__24,24,502,0.7060890,14.21699,1.493443,0.5684205,⋯,96.481507,2,FALSE,FALSE,FALSE,FALSE,TRUE,TRUE,TRUE,FALSE


In [228]:
cat("Intensities dimensions before outlier filtering:", dim(intensities), "\n")
cat("Batches total before filtering:", nrow(final_summary), "\n")
cat("Samples flagged for inspection:", sum(sample_qc$inspect_outlier), "\n")
cat("Samples marked for removal:", sum(sample_qc$remove_outlier), "\n")

outliers <- sample_qc %>%
  filter(remove_outlier) %>%
  pull(file)

intensities <- intensities[, !colnames(intensities) %in% outliers, drop = FALSE]
metadata <- metadata[!metadata$file %in% outliers, , drop = FALSE]

cat("Samples after filtering:", nrow(metadata), "\n")
cat("Intensities dimensions after filtering:", dim(intensities), "\n")


Intensities dimensions before outlier filtering: 1708 228 
Batches total before filtering: 101 


Samples flagged for inspection: 32 
Samples marked for removal: 7 
Samples after filtering: 221 
Intensities dimensions after filtering: 1708 221 


### Per-center NA filtering

Drop features with <20% non-NA values within each lab.
Features present in at least one lab are kept via outer-join.

In [229]:
intensities <- as.data.frame(intensities, check.names = FALSE)

joint_intensities <- NULL
for (center in unique(metadata$lab)) {
    center_metadata <- metadata[metadata$lab == center, ]
    center_intensities <- intensities[, center_metadata$file, drop = FALSE]

    min_non_na <- as.integer(max(1L, ceiling(0.7 * nrow(center_metadata))))
    intens_filtered <- filter_per_center(
        intensities = center_intensities,
        metadata = center_metadata,
        quantitative_column_name = "file",
        centers = unique(center_metadata$lab),
        center_column_name = "lab",
        min_samples = min_non_na,
        drop_row = TRUE
    )

    cat("Center:", center,
        "- removed", nrow(center_intensities) - nrow(intens_filtered),
        "rows with <", min_non_na, "non-NA values\n")

    center_intensities_filtered <- intens_filtered[, center_metadata$file, drop = FALSE]

    if (is.null(joint_intensities)) {
        joint_intensities <- center_intensities_filtered %>%
            tibble::rownames_to_column("rowname")
    } else {
        center_intensities_filtered <- center_intensities_filtered %>%
            tibble::rownames_to_column("rowname")
        joint_intensities <- merge(
            joint_intensities,
            center_intensities_filtered,
            by = "rowname",
            all = TRUE
        )
    }
}

cat("Joint protein intensities:", dim(joint_intensities), "\n")
intensities <- joint_intensities %>% tibble::column_to_rownames("rowname")

Filtering by lab - min 70 not-NA per lab

	Before filtering: 1708 99

	After filtering: 351 99



Center: leduc_plexDIA - removed 1357 rows with < 70 non-NA values


Filtering by lab - min 68 not-NA per lab

	Before filtering: 1708 97

	After filtering: 387 97



Center: derks_QExactive - removed 1321 rows with < 68 non-NA values


Filtering by lab - min 18 not-NA per lab

	Before filtering: 1708 25

	After filtering: 983 25



Center: derks_timsTOFSCP - removed 725 rows with < 18 non-NA values
Joint protein intensities: 1031 222 


### Final cleanup and uncorrected plot

In [230]:
if (!dir.exists("plots")) dir.create("plots")

# Remove remaining all-NA rows/columns
intensities <- intensities[!apply(is.na(intensities), 1, all), , drop = FALSE]
intensities <- intensities[, !apply(is.na(intensities), 2, all), drop = FALSE]
metadata <- metadata[metadata$file %in% colnames(intensities), ]

cat("Final dimensions:", dim(intensities), "\n")
cat("Samples:", nrow(metadata), "\n")

Final dimensions: 1031 221 
Samples: 221 


### Median centering inside each lab

In [231]:
# ## Sample-wise median centering (per lab)
# for (l in unique(metadata$lab)) {
#     lab_samples <- metadata$file[metadata$lab == l]
#     lab_mat <- as.matrix(intensities[, lab_samples, drop = FALSE])
#     sample_medians <- matrixStats::colMedians(lab_mat, na.rm = TRUE)
#     lab_target <- median(sample_medians, na.rm = TRUE)
#     intensities[, lab_samples] <- sweep(lab_mat, 2, sample_medians - lab_target, FUN = "-")
# }

# median centering across all samples (to preserve relative differences between labs)
sample_medians <- matrixStats::colMedians(as.matrix(intensities), na.rm = TRUE)
target <- median(sample_medians, na.rm = TRUE)
intensities <- sweep(as.matrix(intensities), 2, sample_medians - target, FUN = "-")  %>% as.data.frame(check.names = FALSE)


In [232]:
# Save final preprocessed data (uncorrected combined matrix)
saveRDS(intensities, "./before/00_initial_data/combined_proteins_log_matrix.rds")
saveRDS(metadata, "./before/00_initial_data/metadata.rds")

# Also save the full intensities TSV for reference
write.table(intensities %>% as.data.frame() %>% tibble::rownames_to_column("rowname"),
    file = "./before/00_initial_data/combined_proteins_log_matrix.tsv",
    sep = "\t", quote = TRUE, row.names = FALSE)

In [233]:
# Summary by Center
lab_summary2 <- metadata %>%
    dplyr::group_by(lab) %>%
    dplyr::summarise(
        `Total samples` = dplyr::n(),
        `Total features` = sum(rowSums(!is.na(intensities[, file, drop = FALSE])) > 0),
        .groups = "drop"
    )

sample_counts2 <- metadata %>%
    dplyr::group_by(lab, condition) %>%
    dplyr::summarise(count = dplyr::n(), .groups = "drop") %>%
    tidyr::pivot_wider(names_from = condition, values_from = count, values_fill = 0)

final_summary2 <- lab_summary2 %>%
    dplyr::left_join(sample_counts2, by = "lab") %>%
    dplyr::rename(Center = lab) %>%
    dplyr::select(Center, `Total features`, `Total samples`, everything())

print("Dataset Summary by Center:")
knitr::kable(final_summary2, format = "markdown")

[1] "Dataset Summary by Center:"




|Center           | Total features| Total samples| Melanoma| Monocyte| PDAC|
|:----------------|--------------:|-------------:|--------:|--------:|----:|
|derks_QExactive  |            387|            97|       38|       26|   33|
|derks_timsTOFSCP |            983|            25|        9|       10|    6|
|leduc_plexDIA    |            351|            99|       99|        0|    0|

In [234]:
layout <- plots_multiple(intensities, metadata,
    "SCP protein-level, uncorrected (3 labs)", use_nipals = TRUE)
ggsave("plots/protein_uncorrected.png", plot = layout, width = 12, height = 15)

Warning message:
“Removed 138508 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 138508 rows containing non-finite outside the scale range
(`stat_summary()`).”
Warning message:
“Removed 138508 rows containing non-finite outside the scale range
(`stat_density()`).”


In [235]:
# plot the same dataset separately by lab
for (l in unique(metadata$lab)) {
    lab_samples <- metadata$file[metadata$lab == l]
    lab_intensities <- intensities[, lab_samples, drop = FALSE]
    # remove all-NA rows/columns for the lab-specific plot
    lab_intensities <- lab_intensities[!apply(is.na(lab_intensities), 1, all), , drop = FALSE]
    lab_intensities <- lab_intensities[, !apply(is.na(lab_intensities), 2, all), drop = FALSE]

    lab_metadata <- metadata[metadata$file %in% lab_samples, ]
    layout_lab <- plots_multiple(lab_intensities, lab_metadata,
        paste0("SCP protein-level, uncorrected (", l, ")"), use_nipals = TRUE)
    ggsave(paste0("plots/protein_uncorrected_", l, ".png"), plot = layout_lab, width = 12, height = 15)
}

Warning message:
“Removed 3204 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 3204 rows containing non-finite outside the scale range
(`stat_summary()`).”
Warning message:
“Removed 3204 rows containing non-finite outside the scale range
(`stat_density()`).”
Warning message:
“Removed 3061 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 3061 rows containing non-finite outside the scale range
(`stat_summary()`).”
Warning message:
“Removed 3061 rows containing non-finite outside the scale range
(`stat_density()`).”
Warning message:
“Removed 1255 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 1255 rows containing non-finite outside the scale range
(`stat_summary()`).”
Warning message:
“Removed 1255 rows containing non-finite outside the scale range
(`stat_density()`).”


---
# Part 4: Two-Stage limma correction + fedRBE export

Two-stage correction for a realistic federated benchmark, evaluated under **two scenarios**.

### Stage 1 — Within-lab correction (local, no data sharing)
Each lab independently corrects its own internal batch effects, preserving biology (`~ condition`).

### Stage 2 — Across-lab correction (federated via fedRBE)
After within-lab correction, correct for the remaining `lab` effect.

### Two benchmark scenarios

| | Local correction (Stage 1) | Federated correction (Stage 2) |
|---|---|---|
| **Scenario 1** (main, harder) | `Label` + `MedianIntensity` only | `lab` |
| **Scenario 2** (easier, optimistic) | `Set` + `Label` + `MedianIntensity` | `lab` |

**Rationale:** Scenario 2 strips most local nuisance, making the federated problem easier (mostly site-centering). Scenario 1 leaves `Set` effects in, creating a harder and more realistic cross-site integration problem. If the federated app works well in both, that is more credible.

**Decision rule for `Set`:** Correct locally only if (a) multiple Sets exist, (b) each Set has enough samples, (c) condition is not confounded with Set, (d) conditions are replicated across Sets.

In [236]:
## Cross-tabulation of condition x Set per lab
## Check: each Set has both conditions? Enough samples per Set?
for (lab_name in unique(metadata$lab)) {
    lab_meta <- metadata[metadata$lab == lab_name, ]
    cat("\n--- ", lab_name, " ---\n")
    cat("Sets:", length(unique(lab_meta$Set)),
        "  Labels:", length(unique(lab_meta$label)),
        "  Samples:", nrow(lab_meta), "\n")
    print(table(condition = lab_meta$condition, Set = lab_meta$Set))
}


---  leduc_plexDIA  ---
Sets: 45   Labels: 3   Samples: 99 
          Set
condition  wAL_plexMel01 wAL_plexMel03 wAL_plexMel05 wAL_plexMel07
  Melanoma             3             2             3             2
          Set
condition  wAL_plexMel09 wAL_plexMel11 wAL_plexMel13 wAL_plexMel15
  Melanoma             3             2             2             2
          Set
condition  wAL_plexMel17 wAL_plexMel19 wAL_plexMel21 wAL_plexMel23
  Melanoma             3             2             3             3
          Set
condition  wAL_plexMel25 wAL_plexMel27 wAL_plexMel29 wAL_plexMel34
  Melanoma             2             3             2             1
          Set
condition  wAL_plexMel36 wAL_plexMel37 wAL_plexMel38 wAL_plexMel39
  Melanoma             2             1             1             2
          Set
condition  wAL_plexMel40 wAL_plexMel41 wAL_plexMel42 wAL_plexMel43
  Melanoma             3             3             2             2
          Set
condition  wAL_plexMel44 wAL_plexMel4

### Scenario 1 — Main benchmark (harder)
Local: `Label` only (no `Set` correction).  
Federated: `lab`.

In [237]:
## Stage 1: correct Label + MedianIntensity within each lab (no Set)
intens_s1 <- intensities

for (current_lab in unique(metadata$lab)) {
    lab_meta <- metadata[metadata$lab == current_lab, ]
    lab_mat  <- as.matrix(intensities[, lab_meta$file, drop = FALSE])
    #if there is only one condition, do not add the design
    if (length(unique(lab_meta$condition)) > 1) {
        design <- model.matrix(~ condition, data = lab_meta)
    } else {
        design <- NULL
    }

        lab_corr <- limma::removeBatchEffect(
            x          = lab_mat,
            batch      = factor(lab_meta$label),
            design     = if (!is.null(design)) design else NULL
        )
    intens_s1[, lab_meta$file] <- as.data.frame(lab_corr, check.names = FALSE)
    cat("Lab:", current_lab, "- corrected Label\n")
}

## Stage 2: correct lab effect across all labs
design_bio <- model.matrix(~ condition, data = metadata)

intens_s1_corrected <- as.data.frame(
    limma::removeBatchEffect(
        x      = as.matrix(intens_s1),
        batch  = factor(metadata$lab),
        design = design_bio
    ),
    check.names = FALSE
)
cat("\nS1 corrected:", dim(intens_s1_corrected), "\n")

design matrix of interest not specified. Assuming a one-group experiment.



Lab: leduc_plexDIA - corrected Label
Lab: derks_QExactive - corrected Label
Lab: derks_timsTOFSCP - corrected Label


Warning message:
“Partial NA coefficients for 784 probe(s)”



S1 corrected: 1031 221 


In [238]:
# Scenario 1 plots
layout_s1_stage1 <- plots_multiple(intens_s1, metadata,
    "S1: after local correction (Label+MI)", use_nipals = TRUE)
ggsave("plots/s1_after_local_correction.png", plot = layout_s1_stage1, width = 12, height = 15)

layout_s1_full <- plots_multiple(intens_s1_corrected, metadata,
    "S1: after federated correction (lab)", use_nipals = TRUE)
ggsave("plots/s1_after_federated_correction.png", plot = layout_s1_full, width = 12, height = 15)

Warning message:
“Removed 138508 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 138508 rows containing non-finite outside the scale range
(`stat_summary()`).”
Warning message:
“Removed 138508 rows containing non-finite outside the scale range
(`stat_density()`).”
Warning message:
“Removed 138508 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 138508 rows containing non-finite outside the scale range
(`stat_summary()`).”
Warning message:
“Removed 138508 rows containing non-finite outside the scale range
(`stat_density()`).”


In [239]:
# Scenario 1: Export for fedRBE
# Stage-1 corrected data = input for fedRBE. R-corrected = comparison target.
metadata_export_s1 <- metadata
metadata_export_s1$batch <- metadata_export_s1$lab

cat("\n=== Exporting Scenario 1 data for fedRBE ===\n")
export_fedrbe_data(
    intensities = intens_s1,
    metadata = metadata_export_s1,
    intensities_corrected = intens_s1_corrected,
    path_to_data = "before_s1/",
    path_to_after_data = "after_s1/"
)


=== Exporting Scenario 1 data for fedRBE ===
Reference client: leduc_plexDIA 
  Center: derks_QExactive - batches: 1 - reference: FALSE 
  Center: derks_timsTOFSCP - batches: 1 - reference: FALSE 
  Center: leduc_plexDIA - batches: 1 - reference: TRUE 
  datainfo.json written to before_s1/datainfo.json 


### Scenario 2 — Easier benchmark (optimistic)
Local: `Set` + `Label` + `MedianIntensity` (where Set is estimable).
Federated: `lab`.

Uses the diagnostic results to decide per-client whether `Set` is safe to correct.

In [240]:
## Per-lab Set correction flags (adjust after inspecting cross-tabs above)
correct_set_s2 <- c(
    leduc_plexDIA    = TRUE,
    derks_QExactive  = TRUE,
    derks_timsTOFSCP = TRUE
)

## Stage 1: correct Set + Label within each lab
intens_s2 <- intensities

for (current_lab in unique(metadata$lab)) {
    lab_meta <- metadata[metadata$lab == current_lab, ]
    lab_mat  <- as.matrix(intensities[, lab_meta$file, drop = FALSE])

    if (length(unique(lab_meta$condition)) > 1) {
        design <- model.matrix(~ condition, data = lab_meta)
    } else {
        design <- NULL
    }
    do_set <- correct_set_s2[[current_lab]]

    if (do_set) {
        lab_corr <- limma::removeBatchEffect(
            x          = lab_mat,
            batch      = factor(lab_meta$Set),
            batch2     = factor(lab_meta$label),
            design     = if (!is.null(design)) design else NULL
        )
        cat("Lab:", current_lab, "- corrected Set + Label \n")
    } else {
        lab_corr <- limma::removeBatchEffect(
            x          = lab_mat,
            batch      = factor(lab_meta$label),
            design     = if (!is.null(design)) design else NULL
        )
        cat("Lab:", current_lab, "- corrected Label  (Set skipped)\n")
    }
    intens_s2[, lab_meta$file] <- as.data.frame(lab_corr, check.names = FALSE)
}

## Stage 2: correct lab effect across all labs
intens_s2_corrected <- as.data.frame(
    limma::removeBatchEffect(
        x      = as.matrix(intens_s2),
        batch  = factor(metadata$lab),
        design = design_bio
    ),
    check.names = FALSE
)
cat("\nS2 corrected:", dim(intens_s2_corrected), "\n")

design matrix of interest not specified. Assuming a one-group experiment.

Warning message:
“Partial NA coefficients for 246 probe(s)”


Lab: leduc_plexDIA - corrected Set + Label 


Warning message:
“Partial NA coefficients for 220 probe(s)”


Lab: derks_QExactive - corrected Set + Label 


Warning message:
“Partial NA coefficients for 292 probe(s)”


Lab: derks_timsTOFSCP - corrected Set + Label 


Warning message:
“Partial NA coefficients for 784 probe(s)”



S2 corrected: 1031 221 


In [241]:
# Scenario 2 plots
layout_s2_stage1 <- plots_multiple(intens_s2, metadata,
    "S2: after local correction (Set+Label+MI)", use_nipals = TRUE)
ggsave("plots/s2_after_local_correction.png", plot = layout_s2_stage1, width = 12, height = 15)

layout_s2_full <- plots_multiple(intens_s2_corrected, metadata,
    "S2: after federated correction (lab)", use_nipals = TRUE)
ggsave("plots/s2_after_federated_correction.png", plot = layout_s2_full, width = 12, height = 15)

Warning message in nipals::nipals(t(df), ncomp = ncomp, center = TRUE, scale = TRUE, :
“Stopping after 500 iterations for PC 1.
”
Warning message in nipals::nipals(t(df), ncomp = ncomp, center = TRUE, scale = TRUE, :
“Stopping after 500 iterations for PC 1.
”
Warning message:
“Removed 138508 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 138508 rows containing non-finite outside the scale range
(`stat_summary()`).”
Warning message:
“Removed 138508 rows containing non-finite outside the scale range
(`stat_density()`).”
Warning message:
“Removed 138508 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 138508 rows containing non-finite outside the scale range
(`stat_summary()`).”
Warning message:
“Removed 138508 rows containing non-finite outside the scale range
(`stat_density()`).”


In [242]:
# Scenario 2: Export for fedRBE
metadata_export_s2 <- metadata
metadata_export_s2$batch <- metadata_export_s2$lab

cat("\n=== Exporting Scenario 2 data for fedRBE ===\n")
export_fedrbe_data(
    intensities = intens_s2,
    metadata = metadata_export_s2,
    intensities_corrected = intens_s2_corrected,
    path_to_data = "before_s2/",
    path_to_after_data = "after_s2/"
)


=== Exporting Scenario 2 data for fedRBE ===
Reference client: leduc_plexDIA 
  Center: derks_QExactive - batches: 1 - reference: FALSE 
  Center: derks_timsTOFSCP - batches: 1 - reference: FALSE 
  Center: leduc_plexDIA - batches: 1 - reference: TRUE 
  datainfo.json written to before_s2/datainfo.json 


---
# Part 5: Summary

In [243]:
cat("=== Summary ===\n")
cat("Protein features:", nrow(intensities), "\n")
cat("Total samples:", ncol(intensities), "\n")
cat("Labs:", paste(unique(metadata$lab), collapse = ", "), "\n")
cat("Conditions:", paste(unique(metadata$condition), collapse = ", "), "\n")
cat("Aggregation: robustSummary (QFeatures default)\n")
cat("\n--- Scenario 1 (main, harder) ---\n")
cat("  Local: Label \n")
cat("  Federated: lab\n")
cat("  Output: before_s1/ -> after_s1/\n")
cat("\n--- Scenario 2 (easier, optimistic) ---\n")
cat("  Local: Set + Label \n")
cat("  Set flags:", paste(names(correct_set_s2), correct_set_s2,
    sep = "=", collapse = ", "), "\n")
cat("  Federated: lab\n")
cat("  Output: before_s2/ -> after_s2/\n")

=== Summary ===
Protein features: 1031 
Total samples: 221 
Labs: leduc_plexDIA, derks_QExactive, derks_timsTOFSCP 
Conditions: Melanoma, PDAC, Monocyte 
Aggregation: robustSummary (QFeatures default)

--- Scenario 1 (main, harder) ---
  Local: Label 
  Federated: lab
  Output: before_s1/ -> after_s1/

--- Scenario 2 (easier, optimistic) ---
  Local: Set + Label 
  Set flags: leduc_plexDIA=TRUE, derks_QExactive=TRUE, derks_timsTOFSCP=TRUE 
  Federated: lab
  Output: before_s2/ -> after_s2/


# Session info

In [244]:
sessionInfo()

R version 4.5.2 (2025-10-31)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 24.04.4 LTS

Matrix products: default
BLAS/LAPACK: /home/yuliya-cosybio/miniforge3/envs/fedRBE/lib/libopenblasp-r0.3.30.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: Europe/Berlin
tzcode source: system (glibc)

attached base packages:
[1] grid      stats4    stats     graphics  grDevices utils     datasets 
[8] methods   base     

other attached packages:
 [1] data.table_1.18.2.1         viridis_0.6.5              
 [3] viridisLite_0.4.3           ggsci_4.2.0                
 [5] umap_0.2.10.0               gridExtra_2.3              
 [7] patchwork_1.3